# MyPlatform — per-bus capabilities, end to end

This notebook is a guided tour of QProgram's **capability / analyse** layer, told through
one concrete example: an *imaginary* QPU called **MyPlatform** that puts **different
capabilities on different bus types**.

MyPlatform is a flux-tunable transmon device. Each qubit has three buses:

| bus kind  | backed by                  | domains (hw / sw) | capability profile        |
|-----------|----------------------------|-------------------|---------------------------|
| `drive`   | qblox real-time generator  | **hw + sw**       | `qblox-default-v1`        |
| `readout` | qblox real-time generator  | **hw + sw**       | `myplatform-readout-v1`   |
| `flux`    | qdac slow DAC (no FPGA)    | **sw only**       | `myplatform-flux-v1`      |

By the end you will have seen the validator/planner:

1. accept a **drive sweep** as real-time hardware (`[hw|sw]`),
2. **force a flux sweep to software** (`[sw]`) with a `forced-software` warning and reason,
3. **recover hardware averaging** by reordering the loops — manually, then automatically with
   `optimize()`,
4. **reject** a qblox op on the flux bus and a qdac op on the drive bus (hard errors),
5. map every diagnostic back to the offending `.qp` source line,
6. and feed all of that into `execute()` (raise on error, warn on degradation).

Everything below runs against the *real* qprogram core and the *real* qblox/qdac vendor
packages in this monorepo — only the platform that wires them together is invented.

## 1. The capability model in one picture

A platform answers one question for the compiler: *"which DSL features does each part of
my hardware support, and in which execution domain?"* QProgram encodes the answer as a
`PlatformCapabilities`:

```
PlatformCapabilities
├── bus: { (element_kind, bus_kind) -> BusCapabilities }   # per-bus profiles
│        e.g. ("q","drive"), ("q","readout"), ("q","flux")
├── platform: BusCapabilities                              # blocks, sweeps, exprs, bus-less ops
└── default_bus_profile: BusCapabilities                   # fallback for raw-string buses

BusCapabilities(hw, sw)            # the two execution domains; either half may be None
   └── CompilerCapabilities(capabilities: {tokens}, limits: {…}, predicates: (…), …)
```

**Routing** — how each AST node finds its slot (spec §9.3):

* an op that touches a `BusRef` → `bus[(bus.element, bus.kind)]`, else `default_bus_profile`;
* an op that touches a raw string bus → `default_bus_profile`;
* a block / sweep / `set_parameter` / expression → `platform`;
* `expr.*` tokens are **always** checked against `platform`, wherever the op routes.

**hw vs sw** — the required-token set is *domain-agnostic*: the same tokens are checked
against both the `hw` and `sw` halves of the routed slot. Domain-specific behavior comes
from **predicates** that emit a `DomainConstraint` (a soft "this can't run on hw, but sw
dispatch works", attached to a loop *block*) or a `Diagnostic` (a hard "no domain works").

A **profile** (`Profile`) is just a reusable, named bundle of `{tokens} + {limits} +
(predicates)` that can `extend` another profile. A platform decides which profile fills
each slot. That indirection is the whole decoupling story: the core knows nothing about
qblox or qdac; the vendor packages ship profiles; MyPlatform composes them.

## 2. Set up the platform

Importing `my_platform` has three side effects, in order: it activates `qprogram_qblox`
and `qprogram_qdac` (registering their vendor namespaces, ops and profiles), then
registers MyPlatform's two ad-hoc profiles (which `extend` the vendor ones). After that,
`MyPlatform()` is ready and the dynamic `.qblox` / `.qdac` namespaces work on any
`QProgram`.

In [1]:
import warnings

from qprogram import QProgram, dumps, loads, optimize
from qprogram.waveforms import IQDrag, Square
from qprogram.errors import UnsupportedOperationError

from my_platform import MyPlatform   # also activates qblox + qdac, registers ad-hoc profiles

platform = MyPlatform()
schema = platform.get_bus_schema()
q0 = schema.q[0]

print("schema kind :", type(schema).__name__)
print("buses       :", platform.get_buses())
print("drive params:", platform.get_parameters(str(q0.drive)))
print("flux params :", platform.get_parameters(str(q0.flux)))
print("globals     :", platform.get_global_parameters())

schema kind : FluxTunableTransmonSchema
buses       : ['q0/drive', 'q0/readout', 'q0/flux', 'q1/drive', 'q1/readout', 'q1/flux']
drive params: ['lo_frequency', 'gain']
flux params : ['offset', 'dwell']
globals     : ['repetition_duration_ns', 'active_reset']


## 3. Inspect the capabilities, slot by slot

This is where MyPlatform differs from the permissive `ReferencePlatform`: each
`(element, bus_kind)` slot gets its **own** `BusCapabilities`, and the flux slot has
**no hardware half at all** (`hw=None`) because the qdac has no FPGA.

In [2]:
caps = platform.capabilities

print(f"{'slot':18} {'hw profile':26} {'sw profile':26} domains")
print("-" * 86)
for selector, bc in caps.bus.items():
    hw = bc.hw.profile if bc.hw else "—"
    sw = bc.sw.profile if bc.sw else "—"
    print(f"{str(selector):18} {hw:26} {sw:26} {sorted(bc.supported_domains())}")

plat = caps.platform
print(f"{'platform slot':18} {plat.hw.profile:26} {plat.sw.profile:26} {sorted(plat.supported_domains())}")
print(f"\ndefault_bus_profile (raw-string buses): "
      f"{caps.default_bus_profile.hw.profile} {sorted(caps.default_bus_profile.supported_domains())}")

slot               hw profile                 sw profile                 domains
--------------------------------------------------------------------------------------
('q', 'drive')     qblox-default-v1           qblox-default-v1           ['hw', 'sw']
('q', 'readout')   myplatform-readout-v1      myplatform-readout-v1      ['hw', 'sw']
('q', 'flux')      —                          myplatform-flux-v1         ['sw']
platform slot      qprogram-base-v1           qprogram-base-v1           ['hw', 'sw']

default_bus_profile (raw-string buses): qblox-default-v1 ['hw', 'sw']


The ad-hoc profiles tighten one numeric **limit** each, inherited from their vendor
parent via `extends`. And `for_bus()` is the routing function the validator uses — note a
schema-backed `BusRef` lands on its specific slot, while a raw string falls through to the
default.

In [3]:
print("readout min_wait_duration_ns:", caps.for_bus(q0.readout).sw.limits.get("min_wait_duration_ns"),
      "  (vendor qblox default is 4)")
print("flux    min_dwell_ns        :", caps.for_bus(q0.flux).sw.limits.get("min_dwell_ns"),
      "  (vendor qdac default is 100)")
print()
print("for_bus(q0.drive) ->", caps.for_bus(q0.drive).hw.profile, sorted(caps.for_bus(q0.drive).supported_domains()))
print("for_bus(q0.flux)  -> hw:", caps.for_bus(q0.flux).hw,
      " sw:", caps.for_bus(q0.flux).sw.profile, sorted(caps.for_bus(q0.flux).supported_domains()))
print("for_bus('some_raw_bus') ->", caps.for_bus("some_raw_bus").hw.profile, "(fell through to default)")

readout min_wait_duration_ns: 16   (vendor qblox default is 4)
flux    min_dwell_ns        : 200   (vendor qdac default is 100)

for_bus(q0.drive) -> qblox-default-v1 ['hw', 'sw']
for_bus(q0.flux)  -> hw: None  sw: myplatform-flux-v1 ['sw']
for_bus('some_raw_bus') -> qblox-default-v1 (fell through to default)


Here is the actual code that builds that descriptor — the heart of the whole package:

In [4]:
import inspect
print(inspect.getsource(MyPlatform.capabilities.fget))

    @property
    def capabilities(self) -> PlatformCapabilities:
        """The per-bus capability descriptor — the whole point of this example.

        Recomputed on each access (like ``ReferencePlatform``) so that vendor tokens
        registered after platform construction are still picked up.
        """
        # Drive: a qblox real-time waveform generator. Use the vendor profile verbatim;
        # it carries the core bus ops (play/measure/wait/sync/set_*), the qblox waveforms
        # and the vendor.qblox.* ops. Wire it into BOTH domains — qblox can run an op in
        # real time (hw) or step it from software (sw).
        drive = CompilerCapabilities.from_profile("qblox-default-v1")

        # Readout: same qblox generator, but tightened via an ad-hoc profile that raises
        # the minimum `Wait` duration on readout buses (4 ns -> 16 ns). Identical token
        # set, one specialised limit (enforced by the core validator).
        readout = CompilerCapabilities.from_pr

## 4. Example 1 — a real-time Rabi (drive + readout)

A textbook amplitude-Rabi: sweep a drive amplitude inside an averaging block and read
out. Every op touches a **qblox** bus, so every op supports both domains, and the loops
stay real-time hardware. `validate()` returns an empty list; `explain()` renders the plan
with each node's domain in the right-hand column (`[hw|sw]` everywhere).

In [5]:
rabi = QProgram(label="rabi", schema=schema)
amp = rabi.variable("amp", units="a.u.")

with rabi.average(1000), rabi.for_loop(amp, 0.0, 1.0, 0.05):
    rabi.play(q0.drive, IQDrag(amplitude=amp, duration=40, sigma=10, beta=0.5))
    m0 = rabi.measure(q0.readout, "readout_pulse", "weights", name="m0")   # keep the handle

print("diagnostics:", platform.validate(rabi))
print()
print(platform.explain(rabi))

diagnostics: []

plan for 'rabi' — errors: 0 · warnings: 0 · info: 0
body
└─ average 1000:                                                                 [hw|sw]
   └─ for amp in range(0.0, 1.0, 0.05):                                          [hw|sw]
      ├─ play q[0].drive IQDrag(amplitude=amp, duration=40, sigma=10, beta=0.5)  [hw|sw]
      └─ measure q[0].readout "readout_pulse" "weights" name="m0"                [hw|sw]


## 5. Example 2 — a flux sweep is forced to software  ⭐

Now the headline. We sweep a **flux bias** with `qdac.set_offset` inside the loop, while
still playing a drive pulse and reading out.

**What forces the loop to software** is the slot wiring alone:

* `set_offset` routes to the flux slot, which is `BusCapabilities(hw=None, sw=flux)` — so
  the op supports `{sw}` only.
* The `for_loop`'s domain is the intersection of its **op-children's** supports, so it
  collapses to `{sw}` (`set_offset` is `{sw}`, the drive/readout ops are `{hw,sw}`).
* The enclosing `average` then loses `hw` too, by *implicit software-child propagation*:
  a block that contains a software-only sub-block can't itself be real-time.

That demotion produces one `forced-software` **warning** (severity `warning`, not
`error` — the program still runs, just degraded), surfaced on the **outermost** forced
block (`average`, since its parent isn't forced). The drive/readout ops keep their
`[hw|sw]` — only the loop *dispatch* changes.

**The reason names the immediate cause.** The `average` didn't lose `hw` because of
anything *it* directly contains — it lost it because the `for_loop` *inside* it is
software. So its reason is structural: *"contains software-only sub-block 'ForLoop'"*,
with the `for_loop`'s own reason nested for context. (That nested reason comes from the
`qdac-default-v1` predicate, which on seeing a qdac op reference a loop-bound variable
emits a `DomainConstraint` carrying the human-readable text — the slot wiring already
forced the decision; the predicate just *explains* it.)

**And there's an `info` hint.** Look closely: an `average` only ever accumulates
**measurement results**, and the measurement here is on the qblox readout — fully
hardware-capable. The average is software *only* because of how the loops are nested. The
validator spots this and adds a `reorderable-averaging` **info** diagnostic: reorder the
loops and the averaging itself could run in hardware. We do exactly that in §6.

In [6]:
flux_sweep = QProgram(label="flux_spectroscopy", schema=schema)
bias = flux_sweep.variable("bias", units="V")

with flux_sweep.average(500), flux_sweep.for_loop(bias, -0.5, 0.5, 0.05):
    flux_sweep.qdac.set_offset(q0.flux, bias)        # qdac op on the flux bus, sweeping `bias`
    flux_sweep.play(q0.drive, IQDrag(amplitude=0.5, duration=40, sigma=10, beta=0.5))
    flux_sweep.measure(q0.readout, "readout_pulse", "weights", name="m0")

for d in platform.validate(flux_sweep):
    print(f"[{d.severity}] {d.code}: {d.message}")
print()
print(platform.explain(flux_sweep))

[warning] forced-software: Block 'Average' falls back to software execution: contains software-only sub-block 'ForLoop' (qdac.SetOffset references loop-bound variable 'bias'; qdac has no FPGA, so the loop must dispatch from software.).
[info] reorderable-averaging: Block 'Average' runs in software only because it encloses a software sweep; its measurement sequence supports hardware. Moving the sweep outside the average (hoisting the software-only setup with it) would let the averaging run in hardware — see qprogram.optimize().

plan for 'flux_spectroscopy' — errors: 0 · warnings: 1 · info: 1
body
└─ average 500:                                                                  [sw]     ~ forced-sw: contains software-only sub-block 'ForLoop' (qdac.SetOffset references loop-bound variable 'bias'; qdac has no FPGA, so the loop must dispatch from software.)  i reorderable-averaging: Block 'Average' runs in software only because it encloses a software sweep; its measurement sequence supports

### Reading the plan programmatically

`plan()` returns an `ExecutionPlan` — an **identity-keyed** map from each node *instance*
to its `frozenset` of domains. (Identity-keyed because two structurally identical ops must
stay distinct.) Below we walk the program and print each node's planned domain.

In [7]:
plan = platform.plan(flux_sweep)
for node in flux_sweep.body.walk():
    domains = plan.get(node)
    label = type(node).__name__
    print(f"{label:14} -> {sorted(domains) if domains else '—'}")

Block          -> —
Average        -> ['sw']
ForLoop        -> ['sw']
SetOffset      -> ['sw']
Play           -> ['hw', 'sw']
Measure        -> ['hw', 'sw']


## 6. Example 2, continued — recovering hardware averaging

Why could the averaging run in hardware at all? Because an `average` block accumulates
**measurement results** — so only its *averaging-relevant* op-children decide whether the
averaging is a real-time feature. Operations advertise this with a class flag,
`AFFECTS_AVERAGING`, which `MeasurementOperation` (core `measure`, vendor `acquire`) sets
to `True` and everything else leaves `False`. A `set_offset` or `play` inside an average is
repeated, but it doesn't *gate* the average's domain.

In Example 2 the average was software only because it *enclosed* the software flux sweep.
If we move the sweep **outside** the average — and hoist the software-only `set_offset` out
with it — the average then contains only the hardware-capable pulse + measurement, so it
runs in hardware. The program is equivalent (same shots per flux point); only the loop
nesting changed.

In [8]:
reordered = QProgram(label="flux_spectroscopy_reordered", schema=schema)
bias = reordered.variable("bias", units="V")

with reordered.for_loop(bias, -0.5, 0.5, 0.05):     # sweep is now the OUTER loop (software)
    reordered.qdac.set_offset(q0.flux, bias)         # software-only setup, once per flux point
    with reordered.average(500):                     # averaging is now INNER -> hardware
        reordered.play(q0.drive, IQDrag(amplitude=0.5, duration=40, sigma=10, beta=0.5))
        reordered.measure(q0.readout, "readout_pulse", "weights", name="m0")

print("diagnostics:", platform.validate(reordered) or "none")
print()
print(platform.explain(reordered))

diagnostics: none

plan for 'flux_spectroscopy_reordered' — errors: 0 · warnings: 0 · info: 0
body
└─ for bias in range(-0.5, 0.5, 0.05):                                           [sw]
   ├─ qdac.set_offset q[0].flux bias                                             [sw]
   └─ average 500:                                                               [hw|sw]
      ├─ play q[0].drive IQDrag(amplitude=0.5, duration=40, sigma=10, beta=0.5)  [hw|sw]
      └─ measure q[0].readout "readout_pulse" "weights" name="m0"                [hw|sw]


That's exactly the rewrite the `reorderable-averaging` hint suggested — and the free function
`optimize(program, capabilities)` (the rewrite analogue of `validate`) applies it for you. It's
**opt-in** because the reorder groups all shots of a sweep point together instead of interleaving
sweep passes, which differs on a drifting device (the averaged result is identical for a
stationary one).

In [9]:
optimized = optimize(flux_sweep, platform.capabilities)

print("optimized diagnostics:", platform.validate(optimized) or "none")
print()
print(platform.explain(optimized))
print()
print("optimize() reproduced the hand-written reorder:",
      optimized.body == reordered.body)

optimized diagnostics: none

plan for 'flux_spectroscopy' — errors: 0 · warnings: 0 · info: 0
body
└─ for bias in range(-0.5, 0.5, 0.05):                                           [sw]
   ├─ qdac.set_offset q[0].flux bias                                             [sw]
   └─ average 500:                                                               [hw|sw]
      ├─ play q[0].drive IQDrag(amplitude=0.5, duration=40, sigma=10, beta=0.5)  [hw|sw]
      └─ measure q[0].readout "readout_pulse" "weights" name="m0"                [hw|sw]

optimize() reproduced the hand-written reorder: True


## 7. From a diagnostic back to the source line

Every node-bearing diagnostic carries a structural **`path`** (a tuple addressing the node
in the AST). Serialize the program to `.qp` and reload it, and `source_map` translates
that path into a 1-based line number — so a diagnostic points at real text.

In [10]:
diag = platform.validate(flux_sweep)[0]
print("diagnostic path:", diag.path, "  severity:", diag.severity)

qp_text = dumps(flux_sweep)
reloaded = loads(qp_text)            # loads() populates reloaded.source_map
line_no = reloaded.source_map[diag.path]
print(f"maps to .qp line {line_no}\n")

for i, line in enumerate(qp_text.splitlines(), 1):
    marker = "  <-- forced-software here" if i == line_no else ""
    print(f"{i:2} | {line}{marker}")

diagnostic path: (0,)   severity: warning
maps to .qp line 17

 1 | #!QProgram 1.0
 2 | 
 3 | require qdac 0.1
 4 | 
 5 | metadata:
 6 |   label: "flux_spectroscopy"
 7 | 
 8 | schema:
 9 |   element q:
10 |     drive info=IQ
11 |     readout info=IQ+acquires
12 |     flux info=single
13 | 
14 | body:
15 |   var bias units="V"
16 | 
17 |   average 500:  <-- forced-software here
18 |     for bias in range(-0.5, 0.5, 0.05):
19 |       qdac.set_offset q[0].flux bias
20 |       play q[0].drive IQDrag(amplitude=0.5, duration=40, sigma=10, beta=0.5)
21 |       measure q[0].readout "readout_pulse" "weights" name="m0"


## 8. Example 3 — the buses speak different dialects (hard errors)

Capabilities are per-bus, so an op is legal only on a bus whose slot carries its token.
Three ways to get a hard `error` diagnostic:

* a **qblox** vendor op on the **flux** (qdac) bus,
* a **qdac** vendor op on the **drive** (qblox) bus,
* even a **core** `op.set_offset` on the **flux** bus — the qdac profile exposes
  `vendor.qdac.set_offset`, not the core `op.set_offset`, so the core op is unsupported
  there.

Note these are *capability* errors, distinct from build-time errors (e.g. an IQ waveform
on the single-channel flux bus is rejected earlier, when you build the program).

In [11]:
def show_errors(title, build):
    prog = QProgram(label=title, schema=schema)
    build(prog)
    print(f"### {title}")
    diags = platform.validate(prog)
    if not diags:
        print("  (no diagnostics)")
    for d in diags:
        print(f"  [{d.severity}] {d.code}: {d.message}")
    print()

show_errors("qblox_op_on_flux", lambda p: p.qblox.set_markers(q0.flux, "0001"))
show_errors("qdac_op_on_drive", lambda p: p.qdac.set_offset(q0.drive, 0.1))
show_errors("core_set_offset_on_flux", lambda p: p.set_offset(q0.flux, 0.1))

### qblox_op_on_flux
  [error] missing-capability: 'SetMarkers' requires capability 'vendor.qblox.set_markers' which is not supported by 'myplatform-flux-v1' (sw)

### qdac_op_on_drive
  [error] missing-capability: 'SetOffset' requires capability 'vendor.qdac.set_offset' which is not supported by 'qblox-default-v1' (hw) / 'qblox-default-v1' (sw)

### core_set_offset_on_flux
  [error] missing-capability: 'SetOffset' requires capability 'op.set_offset' which is not supported by 'myplatform-flux-v1' (sw)



## 9. Example 4 — the other two capability axes: limits & predicates

Tokens (Example 3) are only one of the three axes a `CompilerCapabilities` carries. The
other two are **limits** (numeric thresholds) and **predicates** (AST-shape checks). Both
are *per-slot*, so MyPlatform tightens them differently on different buses.

**Limit — readout `Wait` duration.** `myplatform-readout-v1` raises `min_wait_duration_ns`
from the qblox default of 4 ns to 16 ns. The core validator enforces this limit directly,
so a sub-16 ns wait on a **readout** bus is an error, while the same wait on a **drive**
bus (limit 4 ns) is fine.

In [12]:
wait_short = QProgram(label="wait_short", schema=schema)
wait_short.wait(q0.readout, 8)        # 8 ns < readout's 16 ns minimum
for d in platform.validate(wait_short):
    print(f"readout wait(8): [{d.severity}] {d.code}: {d.message}")

wait_ok = QProgram(label="wait_ok", schema=schema)
wait_ok.wait(q0.drive, 8)             # 8 ns >= drive's 4 ns minimum -> fine
print("drive   wait(8):", platform.validate(wait_ok) or "OK")

readout wait(8): [error] limit-exceeded: Wait duration 8 ns is shorter than min_wait_duration_ns=16
drive   wait(8): OK


**Predicate — flux dwell.** The core validator has no notion of a DAC "dwell" limit, so
MyPlatform ships its **own predicate** (`_flux_dwell_below_minimum` in `profiles.py`) on
the flux profile. It hard-errors a `qdac.play` whose `dwell` is below MyPlatform's
200 ns floor — and leaves a correctly-dwelled play alone (planning it to `[sw]`, since the
flux bus has no hardware engine). This is the canonical way to enforce a hardware rule the
core knows nothing about.

In [13]:
dwell_short = QProgram(label="flux_dwell_short", schema=schema)
dwell_short.qdac.play(q0.flux, Square(amplitude=0.2, duration=100), dwell=50)
for d in platform.validate(dwell_short):
    print(f"qdac.play(dwell=50):  [{d.severity}] {d.code}: {d.message}")

flux_pulse = QProgram(label="flux_pulse", schema=schema)
flux_pulse.qdac.play(q0.flux, Square(amplitude=0.2, duration=100), dwell=256)
print("qdac.play(dwell=256):", platform.validate(flux_pulse) or "OK")
print()
print(platform.explain(flux_pulse))

qdac.play(dwell=50):  [error] myplatform.flux-dwell-too-short: qdac.play dwell=50 ns is below MyPlatform's flux DAC minimum of 200 ns
qdac.play(dwell=256): OK

plan for 'flux_pulse' — errors: 0 · warnings: 0 · info: 0
body
└─ qdac.play q[0].flux Square(amplitude=0.2, duration=100) dwell=256  [sw]


## 10. Feeding capabilities into `execute()`

`execute()` is where the validation verdict becomes policy. MyPlatform follows the
standard convention: **raise** `UnsupportedOperationError` on any `error`, **warn** (via
`ExecutionWarning`) on a `warning`, and run otherwise. The numeric interpretation itself is
delegated to the reference simulator — but the *legality* decision is made against
MyPlatform's own (narrow) capabilities.

In [14]:
# (a) The clean Rabi runs and returns results.
result = platform.execute(rabi)
print("rabi ->", result)
print()

# (b) The flux sweep runs, but emits an ExecutionWarning (degraded, not illegal).
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    platform.execute(flux_sweep)
print("flux_sweep emitted", len(caught), "warning(s):")
for w in caught:
    print("  ", w.category.__name__, "->", str(w.message).splitlines()[0])
print()

# (c) An illegal program raises before running.
bad = QProgram(label="bad", schema=schema)
bad.qblox.set_markers(q0.flux, "0001")
try:
    platform.execute(bad)
except UnsupportedOperationError as exc:
    print("execute(bad) raised UnsupportedOperationError:")
    print("  ", str(exc).splitlines()[-1])

rabi -> QProgramResult(1 measurements, names=['m0'])

flux_sweep emitted 1 warning(s):
   ExecutionWarning -> [warning] forced-software: Block 'Average' falls back to software execution: contains software-only sub-block 'ForLoop' (qdac.SetOffset references loop-bound variable 'bias'; qdac has no FPGA, so the loop must dispatch from software.). (at body[0])

execute(bad) raised UnsupportedOperationError:
     - [error] missing-capability: 'SetMarkers' requires capability 'vendor.qblox.set_markers' which is not supported by 'myplatform-flux-v1' (sw) (at body[0])


And measurement results come back as labelled `xarray` arrays, dimensioned by the
enclosing sweeps:

In [15]:
data = result.get(m0)               # m0 is the handle captured when we built `rabi`
print("result.get(m0):", type(data).__name__)
print("  dims :", getattr(data, "dims", None))
print("  shape:", getattr(data, "shape", None), " (21 amplitude points x IQ)")

result.get(m0): DataArray
  dims : ('amp', 'IQ')
  shape: (21, 2)  (21 amplitude points x IQ)


## 11. Recap & how to extend

What made the per-bus behavior work — one lever per capability axis:

* **`PlatformCapabilities.bus`** keyed by `(element, bus_kind)` gave each bus its own
  profile — drive/readout on qblox, flux on qdac (the **tokens** axis, Example 3).
* **`BusCapabilities(hw, sw)`** with `hw=None` on flux modeled "this line has no FPGA",
  which is what forced flux loops to software (Example 2).
* **`extends`** let the ad-hoc `myplatform-*-v1` profiles reuse a vendor profile and
  specialise just one thing each: a tighter `Wait` **limit** on readout, and a custom
  dwell **predicate** on flux (Example 4).
* **predicates** played both roles we saw: the inherited qdac predicate supplied the
  *reason* a loop was demoted (Example 2), while MyPlatform's own predicate *enforced* a
  rule the core validator doesn't know about (Example 4) — both surfaced through
  `validate()` / `explain()` / `execute()`.
* **`AFFECTS_AVERAGING`** let the classifier judge an `average` by its measurements alone, so
  a software flux sweep *enclosing* the averaging earned only an advisory
  `reorderable-averaging` hint — and `optimize(program, caps)` turned the demo's flux sweep into
  a hardware-averaged program with one call (§6).

To take it further:

* **Add a coupler flux bus**: switch to `BusSchema.flux_tunable_transmon_coupled()` and add
  a `("c", "flux")` entry to `capabilities.bus`.
* **Make readout real-time-only**: set the readout slot's `sw=None` and watch any
  software-forced readout become a `sw-in-hw` error.
* **Add a new vendor**: ship a package that `register_profile(...)`s its own bundle, then
  reference it from a platform — the core never changes.

See `.specs/qprogram-dsl.md` §9 (capability protocol) and the `CLAUDE.md` "Capability
protocol" section for the normative rules behind everything shown here.